# Reproducing the figures of *Grain boundary segregation of light elements and their effects on cohesion in ferritic steels*

This notebook regenerates every figure that appears in the manuscript and the Supplementary Information from the cached analysis checkpoints in `data/checkpoints/`.

Each section is self-contained: it imports the corresponding script module from `scripts/` and re-runs the figure-generation routine. Outputs are written to `figures/` (overwriting any existing copy) and the produced PNGs are displayed inline.

## Setup

Add the repository scripts directory to `sys.path` so the figure-generation modules are importable. All output directories are resolved relative to the repository root.

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

REPO = Path.cwd()
FIG_DIR = REPO / 'figures'
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'scripts' / 'MainFigures'))
sys.path.insert(0, str(REPO / 'scripts' / 'SupplementaryFigures'))

def show(name, width=600):
    """Display a figure file from the figures/ directory."""
    path = FIG_DIR / name
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'MISSING: {path}')

---
## Main text figures

All main-text figures are produced by a single monolithic script `scripts/MainFigures/generate_main_figures.py`. Running it once generates all per-element panels (8 figures × 4 cohesion/structural quantities), the site-type bar chart (Fig. 9), the segregation histograms, and the min-Eseg-vs-GB-energy plot (Fig. 3).

### Generate all main-text figures

In [ ]:
import subprocess
result = subprocess.run([sys.executable, str(REPO / 'scripts' / 'MainFigures' / 'generate_main_figures.py')],
                        capture_output=True, text=True, check=True)
print(result.stdout.split('All figures saved')[0][-400:])
print('Done.')

### Fig. 3 — Minimum segregation energy versus GB energy, per element

In [ ]:
show('min_eseg_vs_gb_energy_per_element.png')

### Fig. 4 — Segregation energy versus distance from the GB plane
Eight per-element panels.

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_dist_GB_{el}.png', width=400)

### Fig. 5 — Segregation energy versus Voronoi volume

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_Voronoi_NN_dist_{el}.png', width=400)

### Fig. 6 — Segregation energy versus minimum nearest-neighbour distance

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_min_nn_dist_{el}.png', width=400)

### Fig. 7 — Rice–Wang work-of-separation cohesion engineering maps

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_RWsepRGS_{el}.png', width=400)

### Fig. 8 — ANSBO cohesion engineering maps

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_R_ANSBO_{el}.png', width=400)

### Fig. 9 — Site type classification (interstitial / substitutional / mixed)

In [ ]:
show('element_site_types.png', width=700)

### Per-element segregation energy histograms (split by site type)

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'Eseg_histogram_{el}.png', width=400)

---
## Supplementary Information figures

Each SI figure (or figure group) is produced by a dedicated script in `scripts/SupplementaryFigures/`. The scripts can be run individually.

### SI Fig. 1 — KP vs KS segregation-energy parity plot

Compares segregation energies computed with the explicit Γ-centred KPOINTS mesh versus the auto KSPACING-generated mesh, across 4{,}481 matched calculations. Also generates the un-zoomed $\Delta E_\mathrm{seg}$ histogram and per-element box plot (Fig. S2 panels (a) and (b)).

In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts' / 'SupplementaryFigures' / 'generate_SI_kpoint_parity.py')],
               check=True)
show('SI_KP_vs_KS_Eseg_scatter.png')
display(Markdown('**$\\Delta E_\\mathrm{seg}$ histogram (Fig. S2a) and per-element box plot (Fig. S2b):**'))
show('SI_KP_vs_KS_dEseg_histogram.png', width=500)
show('SI_KP_vs_KS_dEseg_boxplot.png', width=500)

### SI Fig. 2 (zoom panels) — ±0.2 eV restricted distribution

After excluding the small number of structural outliers, restricts to $|\Delta E_\mathrm{seg}| \le 0.2$ eV, retaining 99.8% of matched calculations.

In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts' / 'SupplementaryFigures' / 'generate_SI_kpoint_zoom.py')],
               check=True)
show('SI_KP_vs_KS_dEseg_histogram_filt_pm02.png', width=500)
show('SI_KP_vs_KS_dEseg_boxplot_filt_pm02.png', width=500)

### SI Fig. 3 — Multi-shell nearest-neighbour correlation

Spearman rank correlation $\rho_\mathrm{full}$ between segregation energy and the $k$-th nearest-neighbour distance for $k=1,\ldots,6$, plotted as a function of shell index for each light-element solute.

In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts' / 'SupplementaryFigures' / 'generate_SI_nn_correlation.py')],
               check=True)
show('SI_rho_vs_kNN_shell.png', width=600)

### SI Fig. 4 — Starting (unrelaxed) Voronoi volume vs. relaxed segregation energy

Eight per-element scatter panels, demonstrating that the starting Voronoi volume is not a reliable predictor of the relaxed segregation energy. These figures are produced by the main figure script (they share the same data pipeline).

In [ ]:
for el in ['H','He','B','C','N','O','P','S']:
    show(f'SI_StartVol_vs_Eseg_{el}.png', width=400)

### SI Fig. 5 — GB vacancy stability

Per-site relaxed vacancy formation energies $E_\mathrm{vf}$ at Fe sites within the segregation zone of each CSL GB. All 56 retained sites yield positive $E_\mathrm{vf}$, confirming the pure GB reference structures are stable against spontaneous vacancy formation. Also writes the LaTeX summary and per-site tables to `data/SI_vacancy_*.tex`.

In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts' / 'SupplementaryFigures' / 'generate_SI_vacancy_tables.py')],
               check=True)
show('SI_vacancy_formation_by_GB.png', width=600)